# Reading member particle IDs (`galaxy_pid_NNNN.pkl` / `.txt`)

If you request member particles, the download also has a `galaxy_pid_NNNN` file for each snapshot. It lists the IDs of the particles that belong to each galaxy.

| File | What it holds | Notebook |
|---|---|---|
| `galaxy_NNNN.pkl` or `.txt` (`halo_NNNN` for halos) | One row per object, properties as columns | `01_catalog.ipynb` |
| `galaxy_pid_NNNN.pkl` or `.txt` | Member particle IDs of each object (only if requested) | `02_members.ipynb` |
| `snapinf_NNNN.txt` | Snapshot metadata (`aexp`, aperture lists) | `03_snapinfo.ipynb` |
| `error.txt` | Requested snapshots that could not be produced (only if any) | `03_snapinfo.ipynb` |

These files store every member particle ID, so they are much larger than the catalog. For example, NewCluster snapshot 700 has about 650 million member particles in total, which is about 5 GB in memory. Section 4 builds one more array of that size. This notebook needs Python 3.7+, NumPy and `ndps_reader.py`, which sits one folder up.

In [ ]:
import sys
sys.path.insert(0, '..')   # ndps_reader.py is one folder up; not needed if you pip-installed it

from pathlib import Path
import zipfile
import numpy as np
import ndps_reader as nr

print('ndps_reader', nr.__version__)

## 1. Unzip the download

In [ ]:
ZIP_FILE = Path('20261103_000000.zip')   # <- a download made with member particles

def extract(zip_file):
    """Unzip a download once, into a folder named after it."""
    folder = zip_file.with_suffix('')
    if not folder.exists():
        with zipfile.ZipFile(zip_file) as archive:
            archive.extractall(folder)
    return folder

DATA_DIR = extract(ZIP_FILE)
files = sorted(p.name for p in DATA_DIR.iterdir())
print(len(files), 'files:', files[:4], '...', files[-2:])

## 2. Read one snapshot

In [ ]:
SNAP = 700
pid = nr.read_members(DATA_DIR / f'galaxy_pid_{SNAP:04d}.pkl')
print(pid.shape, pid.dtype.names)

There is one row per galaxy, with three fields:

| Field | Contents |
|---|---|
| `ID` | Galaxy ID, the same as `ID` in `galaxy_NNNN` |
| `Npart` | Number of member particles |
| `P_ID` | Array of the member particle IDs, `Npart` long |

Each galaxy has a different number of particles, so each `P_ID` entry is a separate NumPy array.

In [ ]:
first = pid[0]
print('ID', first['ID'], '  Npart', first['Npart'])
print('first particle IDs:', first['P_ID'][:10])
print(type(first['P_ID']).__name__, first['P_ID'].dtype)

## 3. Match with the catalog

Match rows by `ID` rather than by position:

In [ ]:
cat = nr.read_catalog(DATA_DIR / f'galaxy_{SNAP:04d}.pkl')
pid_of = {id_: p for id_, p in zip(pid['ID'], pid['P_ID'])}

g = cat[np.argmax(cat['Mass_tot'])]
particles = pid_of[g['ID']]
print(f"most massive galaxy, ID {g['ID']}: {len(particles)} particles (catalog npart = {g['npart']})")

The particles of every galaxy above a mass cut:

In [ ]:
massive_ids = cat['ID'][cat['Mass_tot'] > 1e11]
massive_particles = np.concatenate([pid_of[i] for i in massive_ids])
print(len(massive_ids), 'galaxies,', len(massive_particles), 'particles')

## 4. Which galaxy does a particle belong to?

Join all the IDs into one array, with the owning galaxy's ID alongside each particle:

In [ ]:
all_pids = np.concatenate(pid['P_ID'])
owner = np.repeat(pid['ID'], pid['Npart'])
print(len(all_pids), 'particle IDs in', len(pid), 'galaxies')

query = particles[:3]                       # a few particle IDs to look up
hit = np.isin(all_pids, query)
for p, o in zip(all_pids[hit], owner[hit]):
    print(f'particle {p} -> galaxy {o}')

For many lookups, sort once and use `searchsorted`. Particles that are in no galaxy get -1:

In [ ]:
order = np.argsort(all_pids)
sorted_pids = all_pids[order]

def owner_of(ids):
    ids = np.asarray(ids)
    i = np.minimum(np.searchsorted(sorted_pids, ids), len(sorted_pids) - 1)
    return np.where(sorted_pids[i] == ids, owner[order[i]], -1)

owner_of(list(query) + [-12345])

## 5. The ASCII version (`.txt`)

Each line holds `ID Npart P_ID...`, separated by spaces, after a `#` header line. `read_members` reads it into the same three fields.

In [ ]:
TXT_ZIP = Path('20261103_000001.zip')   # <- an ASCII download with member particles, or None to skip

if TXT_ZIP is not None:
    path = extract(TXT_ZIP) / f'galaxy_pid_{SNAP:04d}.txt'
    with open(path) as f:
        print(f.readline().rstrip())
        print(f.readline()[:70], '...')
    pid_txt = nr.read_members(path)
    same = (np.array_equal(pid['ID'], pid_txt['ID'])
            and all(np.array_equal(a, b) for a, b in zip(pid['P_ID'], pid_txt['P_ID'])))
    print('same as the pickle:', same)

## Notes

- The ASCII reader always returns int64 IDs. A pickle keeps the type it was written with.
- Memory use is about 8 bytes per particle ID:

In [ ]:
print(f"{sum(p.nbytes for p in pid['P_ID']) / 1e6:.1f} MB of particle IDs")